# Model 1: 1D CNN Fingerprint Predictor

## 1. Environment Setup and Data Loading

This notebook implements a 1D CNN baseline for predicting
molecular fingerprints from MS/MS spectra.

### Model Configuration
- Input: 2200-dimensional binned MS/MS spectrum
- Target: 1024-bit Morgan fingerprint (radius=2)
- Training data: 17,456 prototype spectra
- Validation data: 2,000 prototype spectra

The datasets were prepared in `02_preprocessing.ipynb`
using a scoring-key-aware train/validation split.

We first verify the saved datasets before building the CNN.

In [ ]:
!pip -q install "rdkit==2026.3.3" pyarrow

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rdkit import rdBase


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current

# Set these environment variables to override the local defaults.
REPO_ROOT = Path(
    os.environ.get("CASMI_REPO_ROOT", find_repo_root())
).expanduser().resolve()
SAVE_DIR = Path(
    os.environ.get(
        "CASMI_MEMBER2_DIR",
        REPO_ROOT / "outputs" / "member2"
    )
).expanduser().resolve()
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = SAVE_DIR / "prototype_data"
print("Member 2 working directory:", SAVE_DIR)

# Load prototype datasets
df_train = pd.read_parquet(
    DATA_DIR / "train_prototype.parquet"
)

df_val = pd.read_parquet(
    DATA_DIR / "val_prototype.parquet"
)

# Load finalized structure split
with open(SAVE_DIR / "structure_split_scoring_safe.json") as f:
    split_info = json.load(f)

train_ids = set(split_info["train_structures"])
val_ids = set(split_info["val_structures"])

# Display basic information
print("PyTorch version:", torch.__version__)
print("RDKit version:", rdBase.rdkitVersion)
print("CUDA available:", torch.cuda.is_available())

print("\nTraining spectra:", len(df_train))
print("Validation spectra:", len(df_val))

print("\nTraining columns:")
print(df_train.columns.tolist())

# Validate saved datasets
assert rdBase.rdkitVersion == "2026.03.3"

assert len(df_train) == 17456
assert len(df_val) == 2000

assert set(df_train["inchikey14"]).issubset(train_ids)
assert set(df_val["inchikey14"]).issubset(val_ids)

assert set(df_train["inchikey14"]).isdisjoint(
    set(df_val["inchikey14"])
)

print("\nPrototype datasets loaded successfully!")

## 2. PyTorch Dataset and DataLoader

We convert MS/MS spectra and molecular structures into
PyTorch tensors for CNN training.

### Input
- 2200-dimensional binned MS/MS spectrum
- Bin width: 1 Da
- Maximum intensity aggregation
- Square-root intensity transformation
- Tensor shape: (1, 2200)

### Target
- 1024-bit Morgan fingerprint
- Radius: 2
- Chirality: disabled
- Binary vector representing molecular structure

Spectrum binning is performed on demand.
Molecular fingerprints are cached to reduce repeated
RDKit computations.

The same preprocessing will be used for both CNN models.

In [ ]:
from functools import lru_cache

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Configuration
MZ_MAX = 2200
BIN_WIDTH = 1.0

FP_SIZE = 1024
FP_RADIUS = 2

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS,
    fpSize=FP_SIZE,
    includeChirality=False
)


def spectrum_to_bins(mzs, intensities):
    """Convert an MS/MS spectrum into a fixed-length vector."""

    mzs = np.asarray(mzs, dtype=np.float64)
    intensities = np.asarray(intensities, dtype=np.float32)

    if len(mzs) != len(intensities):
        raise ValueError("Mismatched spectrum array lengths")

    vector = np.zeros(MZ_MAX, dtype=np.float32)

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs >= 0)
        & (mzs < MZ_MAX)
        & (intensities >= 0)
    )

    indices = np.floor(
        mzs[valid] / BIN_WIDTH
    ).astype(np.int64)

    values = np.sqrt(intensities[valid])

    np.maximum.at(vector, indices, values)

    return vector


@lru_cache(maxsize=30000)
def smiles_to_fingerprint(smiles):
    """Convert SMILES into a 1024-bit Morgan fingerprint."""

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    fp = morgan_generator.GetFingerprint(mol)

    arr = np.zeros(FP_SIZE, dtype=np.uint8)

    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr


print("Preprocessing functions defined successfully!")

In [ ]:
from torch.utils.data import DataLoader, Dataset


class SpectrumFingerprintDataset(Dataset):

    def __init__(self, dataframe):
        self.records = dataframe.to_dict("records")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]

        spectrum = spectrum_to_bins(
            row["ms2_mzs"],
            row["ms2_normalized_intensities"]
        )

        fingerprint = smiles_to_fingerprint(
            row["normalized_smiles"]
        )

        x = torch.from_numpy(spectrum).unsqueeze(0)

        y = torch.from_numpy(
            fingerprint.astype(np.float32)
        )

        return x, y


# Create datasets
train_dataset = SpectrumFingerprintDataset(df_train)
val_dataset = SpectrumFingerprintDataset(df_val)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
X_batch, y_batch = next(iter(train_loader))

print("Input shape:", X_batch.shape)
print("Target shape:", y_batch.shape)

print("\nInput dtype:", X_batch.dtype)
print("Target dtype:", y_batch.dtype)

print("\nInput range:",
      X_batch.min().item(),
      X_batch.max().item())

print("\nTarget unique values:", torch.unique(y_batch))

assert X_batch.shape == (32, 1, 2200)
assert y_batch.shape == (32, 1024)

assert torch.isfinite(X_batch).all()
assert torch.isfinite(y_batch).all()

assert torch.all((y_batch == 0) | (y_batch == 1))

print("\nAll DataLoader tests passed!")

## 3. 1D CNN Baseline Architecture

We build a 1D CNN to predict molecular fingerprints from binned MS/MS spectra.

### Architecture

1. Conv1d (1 → 32 channels, kernel size 7)
2. BatchNorm + ReLU + MaxPooling
3. Conv1d (32 → 64 channels, kernel size 5)
4. BatchNorm + ReLU + MaxPooling
5. Conv1d (64 → 128 channels, kernel size 3)
6. BatchNorm + ReLU
7. Adaptive Average Pooling
8. Fully Connected Layers
9. Output: 1024 fingerprint logits

The CNN extracts spectral patterns at different levels, while the fully connected layers map the learned features to molecular fingerprint predictions.

We use raw logits as the output because the model will be trained with BCEWithLogitsLoss.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)


class CNN1DFingerprint(nn.Module):

    def __init__(self, fp_size=1024):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv1d(
                in_channels=1,
                out_channels=32,
                kernel_size=7,
                padding=3
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            # Block 2
            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            # Block 3
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            # Reduce sequence length
            nn.AdaptiveAvgPool1d(16)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, fp_size)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = CNN1DFingerprint(
    fp_size=FP_SIZE
).to(device)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Device:", device)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\nModel architecture:")
print(model)

In [ ]:
# Test forward pass with one batch

model.eval()

X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

with torch.no_grad():
    logits = model(X_batch)
    probabilities = torch.sigmoid(logits)

print("Input shape:", X_batch.shape)
print("Target shape:", y_batch.shape)
print("Output logits shape:", logits.shape)

print("\nProbability range:")
print(
    probabilities.min().item(),
    probabilities.max().item()
)

print("\nOutput contains NaN:",
      torch.isnan(logits).any().item())

print("Output contains Inf:",
      torch.isinf(logits).any().item())

# Shape and numerical checks
assert logits.shape == y_batch.shape
assert torch.isfinite(logits).all()
assert probabilities.min() >= 0
assert probabilities.max() <= 1

print("\nCNN forward pass test passed!")

## 4. Loss Function and Optimizer

We use BCEWithLogitsLoss because molecular fingerprint
prediction is a multi-label binary classification task.

Each of the 1024 fingerprint bits is predicted independently.

Since most fingerprint bits are zero, we calculate positive
class weights from the training data to reduce the effect
of class imbalance.

We use:
- Loss: BCEWithLogitsLoss with positive class weights
- Optimizer: AdamW
- Learning rate: 0.001
- Weight decay: 0.0001

Positive class weights are capped at 20 to avoid
extremely large weights for rare fingerprint bits.

The weighting strategy will be evaluated against an
unweighted baseline in later experiments.

In [ ]:
from tqdm.auto import tqdm

# Count positive fingerprint bits in training data
bit_counts = np.zeros(FP_SIZE, dtype=np.int64)

for smiles in tqdm(
    df_train["normalized_smiles"],
    desc="Calculating fingerprint frequencies"
):
    fp = smiles_to_fingerprint(smiles)
    bit_counts += fp.astype(np.int64)

n_train = len(df_train)

# Calculate bit frequencies
bit_prevalence = bit_counts / n_train

print("Training spectra:", n_train)
print("Average active bits:", bit_counts.sum() / n_train)

print("Bits never activated:", np.sum(bit_counts == 0))
print("Bits activated at least once:", np.sum(bit_counts > 0))

# Calculate positive weights
pos_weights = np.ones(FP_SIZE, dtype=np.float32)

seen = bit_counts > 0

pos_weights[seen] = np.clip(
    (n_train - bit_counts[seen]) / bit_counts[seen],
    1.0,
    20.0
)

pos_weight_tensor = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=device
)

print("\nPositive weight range:",
      pos_weight_tensor.min().item(),
      pos_weight_tensor.max().item())

print("\nClass weights calculated successfully!")

In [ ]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("Loss:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])

print("\nTraining configuration ready!")

In [ ]:
model.train()

X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

# Save weights before optimization
weights_before = (
    model.classifier[-1].weight.detach().clone()
)

# Forward pass
optimizer.zero_grad(set_to_none=True)

logits = model(X_batch)

# Compute loss
loss = criterion(logits, y_batch)

print("Initial training loss:", loss.item())

# Backward pass
loss.backward()

gradient_norm = (
    model.classifier[-1].weight.grad.norm().item()
)

print("Gradient norm:", gradient_norm)

# Update parameters
optimizer.step()

# Verify parameter update
weight_change = (
    model.classifier[-1].weight.detach()
    - weights_before
).abs().mean().item()

print("Average weight change:", weight_change)

assert torch.isfinite(loss)
assert gradient_norm > 0
assert weight_change > 0

print("\nForward, backward, and optimizer step passed!")

## 5. Training and Evaluation

We implement reusable training and evaluation functions
for the 1D CNN fingerprint predictor.

We monitor:
- Weighted BCE loss
- Micro precision
- Micro recall
- Micro F1 score
- Mean fingerprint Tanimoto similarity
- Predicted positive-bit rate

Since molecular fingerprints are sparse, monitoring
loss alone is not sufficient.

We first run a small smoke test to verify the training
pipeline before full model training.

In [ ]:
from tqdm.auto import tqdm


def run_epoch(model, loader, criterion, optimizer=None,
              max_batches=None):

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_samples = 0

    tp = 0
    fp = 0
    fn = 0

    predicted_positive = 0
    actual_positive = 0
    total_bits = 0

    tanimoto_sum = 0.0

    for batch_idx, (x, y) in enumerate(loader):

        if max_batches is not None and batch_idx >= max_batches:
            break

        x = x.to(device)
        y = y.to(device)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):

            logits = model(x)
            loss = criterion(logits, y)

            if is_training:
                loss.backward()
                optimizer.step()

        batch_size = x.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        # Fingerprint metrics
        with torch.no_grad():

            probs = torch.sigmoid(logits)

            preds = probs >= 0.5
            targets = y >= 0.5

            tp += (preds & targets).sum().item()
            fp += (preds & ~targets).sum().item()
            fn += (~preds & targets).sum().item()

            predicted_positive += preds.sum().item()
            actual_positive += targets.sum().item()
            total_bits += targets.numel()

            # Per-spectrum Tanimoto similarity
            intersection = (
                preds & targets
            ).sum(dim=1).float()

            union = (
                preds | targets
            ).sum(dim=1).float()

            tanimoto = torch.where(
                union > 0,
                intersection / union.clamp(min=1),
                torch.ones_like(union)
            )

            tanimoto_sum += tanimoto.sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return {
        "loss": total_loss / total_samples,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tanimoto": tanimoto_sum / total_samples,
        "pred_positive_rate": predicted_positive / total_bits,
        "actual_positive_rate": actual_positive / total_bits
    }


print("Training and evaluation functions ready!")

In [ ]:
# Reset model for reproducible training
torch.manual_seed(42)

model = CNN1DFingerprint(
    fp_size=FP_SIZE
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

print("Running training smoke test...")

# Train on only 5 batches
train_metrics = run_epoch(
    model,
    train_loader,
    criterion,
    optimizer=optimizer,
    max_batches=5
)

# Validate on only 3 batches
val_metrics = run_epoch(
    model,
    val_loader,
    criterion,
    optimizer=None,
    max_batches=3
)

print("\nTraining Metrics:")
for key, value in train_metrics.items():
    print(f"{key}: {value:.4f}")

print("\nValidation Metrics:")
for key, value in val_metrics.items():
    print(f"{key}: {value:.4f}")

assert np.isfinite(train_metrics["loss"])
assert np.isfinite(val_metrics["loss"])

print("\nTraining smoke test passed!")

## 6. Prototype Training

The training smoke test passed.

However, the model currently predicts more positive
fingerprint bits than the actual target distribution.

We will run a longer training experiment to observe
whether the model improves as training progresses.

We will monitor:
- Training and validation loss
- Fingerprint micro F1
- Tanimoto similarity
- Predicted positive-bit rate

Since the current environment only has CPU,
we first use a limited number of training batches.
Full training will be performed separately.

In [ ]:
import time

import pandas as pd
import torch
import torch.nn as nn

torch.manual_seed(42)

# Reset model
model = CNN1DFingerprint(
    fp_size=FP_SIZE
).to(device)

# Loss function
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor.to(device)
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# Training configuration
EPOCHS = 3
MAX_TRAIN_BATCHES = 100  # CPU prototype only

history = []

print("Device:", device)
print("Epochs:", EPOCHS)
print("Training batches per epoch:", MAX_TRAIN_BATCHES)
print("Validation batches:", len(val_loader))

print("\nModel reset successfully!")

In [ ]:
for epoch in range(1, EPOCHS + 1):

    start_time = time.perf_counter()

    # Training
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        max_batches=MAX_TRAIN_BATCHES
    )

    # Validation
    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
        max_batches=None
    )

    elapsed = time.perf_counter() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_f1": train_metrics["f1"],
        "val_loss": val_metrics["loss"],
        "val_f1": val_metrics["f1"],
        "val_tanimoto": val_metrics["tanimoto"],
        "val_pred_positive_rate": val_metrics["pred_positive_rate"],
        "val_actual_positive_rate": val_metrics["actual_positive_rate"],
        "seconds": elapsed
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Tanimoto: {val_metrics['tanimoto']:.4f} | "
        f"Pred Positive: {val_metrics['pred_positive_rate']:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

history_df = pd.DataFrame(history)

print("\nTraining Summary:")
display(history_df)

## 7. Full Prototype Training with Early Stopping

The initial three-epoch experiment showed that
training loss decreased, while validation loss
increased after the first epoch.

To improve training stability, we will:

- Reduce the learning rate to 0.0003
- Use all training batches in each epoch
- Evaluate on the complete validation subset
- Apply early stopping based on validation loss
- Save the best model checkpoint

This experiment uses the 17,456-spectrum prototype
training dataset, not the full competition dataset.

In [ ]:
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Reinitialize model
model = CNN1DFingerprint(
    fp_size=FP_SIZE
).to(device)

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor.to(device)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# Training settings
EPOCHS = 8
PATIENCE = 3

best_val_loss = float("inf")
epochs_without_improvement = 0

history_full = []

# Checkpoint directory
CKPT_DIR = SAVE_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = CKPT_DIR / "cnn1d_best_val_loss.pt"

print("Starting full prototype training...")
print("Device:", device)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

for epoch in range(1, EPOCHS + 1):

    start_time = time.perf_counter()

    # Full training epoch
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        max_batches=None
    )

    # Full validation
    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
        max_batches=None
    )

    elapsed = time.perf_counter() - start_time

    history_full.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_f1": train_metrics["f1"],
        "val_loss": val_metrics["loss"],
        "val_f1": val_metrics["f1"],
        "val_tanimoto": val_metrics["tanimoto"],
        "val_pred_positive_rate": val_metrics["pred_positive_rate"],
        "seconds": elapsed
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Tanimoto: {val_metrics['tanimoto']:.4f} | "
        f"Pred Positive: {val_metrics['pred_positive_rate']:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    # Save best model
    if val_metrics["loss"] < best_val_loss - 1e-4:

        best_val_loss = val_metrics["loss"]
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_metrics["loss"],
            "val_f1": val_metrics["f1"],
            "val_tanimoto": val_metrics["tanimoto"],
            "fp_size": FP_SIZE,
            "mz_max": MZ_MAX
        }, best_model_path)

        print("  -> Best model saved!")

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping triggered.")
            break

# Save history
history_full_df = pd.DataFrame(history_full)

history_path = SAVE_DIR / "cnn1d_training_history.csv"
history_full_df.to_csv(history_path, index=False)

print("\nTraining Summary:")
display(history_full_df)

print("\nBest validation loss:", best_val_loss)
print("Best model:", best_model_path)
print("Training history:", history_path)

## 8. Best Checkpoint Evaluation and Threshold Analysis

The full prototype training completed successfully.

The model achieved its lowest validation loss at epoch 8.
However, the predicted positive-bit rate was higher
than the actual fingerprint positive-bit rate.

We will load the best checkpoint and evaluate different
probability thresholds.

Metrics:
- Micro Precision
- Micro Recall
- Micro F1
- Mean Tanimoto Similarity
- Predicted Positive-Bit Rate

This analysis uses only the validation set.
The final candidate-ranking performance will be
evaluated separately using MRR@25.

In [ ]:
# Load the best CNN checkpoint
checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True
)

best_model = CNN1DFingerprint(
    fp_size=FP_SIZE
).to(device)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_model.eval()

print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation Loss:", checkpoint["val_loss"])
print("Validation F1:", checkpoint["val_f1"])
print("Validation Tanimoto:", checkpoint["val_tanimoto"])

print("\nBest checkpoint loaded successfully!")

In [ ]:
best_model.eval()

all_probs = []
all_targets = []

with torch.inference_mode():

    for x, y in tqdm(
        val_loader,
        desc="Evaluating best checkpoint"
    ):

        x = x.to(device)

        logits = best_model(x)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu())
        all_targets.append(y.cpu())

val_probs = torch.cat(all_probs, dim=0)
val_targets = torch.cat(all_targets, dim=0)

print("Predictions shape:", val_probs.shape)
print("Targets shape:", val_targets.shape)

assert val_probs.shape == (2000, 1024)
assert val_targets.shape == (2000, 1024)

print("\nValidation predictions ready!")

In [ ]:
threshold_results = []

targets = val_targets >= 0.5

thresholds = np.round(
    np.arange(0.20, 0.81, 0.05),
    2
)

for threshold in thresholds:

    preds = val_probs >= threshold

    tp = (preds & targets).sum().item()
    fp = (preds & ~targets).sum().item()
    fn = (~preds & targets).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    # Per-spectrum Tanimoto
    intersection = (
        preds & targets
    ).sum(dim=1).float()

    union = (
        preds | targets
    ).sum(dim=1).float()

    tanimoto = torch.where(
        union > 0,
        intersection / union.clamp(min=1),
        torch.ones_like(union)
    ).mean().item()

    pred_positive_rate = preds.float().mean().item()

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tanimoto": tanimoto,
        "pred_positive_rate": pred_positive_rate
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(4))

best_f1_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

best_tanimoto_row = threshold_df.loc[
    threshold_df["tanimoto"].idxmax()
]

print("\nBest threshold by F1:")
print(best_f1_row.round(4))

print("\nBest threshold by Tanimoto:")
print(best_tanimoto_row.round(4))

## 9. Save Baseline Evaluation Results

The threshold analysis showed that the default
threshold of 0.5 was not optimal for fingerprint
prediction.

The best validation Micro F1 was 0.3163 at
threshold 0.65, while the best mean Tanimoto
similarity was 0.1863 at threshold 0.60.

We save:
- Threshold comparison results
- Validation prediction probabilities
- Validation fingerprint targets
- Best threshold configurations

The continuous fingerprint probabilities will be
used later for candidate molecular ranking.

These results are based on the prototype validation
set and should not be considered final competition
performance.

In [ ]:
import json
from pathlib import Path

EVAL_DIR = SAVE_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save threshold analysis
threshold_path = EVAL_DIR / "cnn1d_threshold_results.csv"

threshold_df.to_csv(
    threshold_path,
    index=False
)

# 2. Save continuous probabilities and targets
# Row order matches df_val because val_loader uses shuffle=False.
predictions_path = EVAL_DIR / "cnn1d_val_predictions.npz"

assert len(val_probs) == len(df_val)
assert len(val_targets) == len(df_val)

np.savez_compressed(
    predictions_path,
    probabilities=val_probs.numpy().astype(np.float32),
    targets=val_targets.numpy().astype(np.uint8),
    inchikey14=df_val["inchikey14"].to_numpy(dtype=str),
    smiles=df_val["normalized_smiles"].to_numpy(dtype=str)
)

# 3. Save evaluation summary
evaluation_summary = {
    "model": "CNN1DFingerprint",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "validation_samples": len(df_val),
    "fingerprint_size": FP_SIZE,
    "best_val_loss": float(checkpoint["val_loss"]),
    "best_f1_threshold": float(best_f1_row["threshold"]),
    "best_f1": float(best_f1_row["f1"]),
    "best_tanimoto_threshold": float(
        best_tanimoto_row["threshold"]
    ),
    "best_tanimoto": float(best_tanimoto_row["tanimoto"]),
    "ranking_input": "continuous fingerprint probabilities"
}

summary_path = EVAL_DIR / "cnn1d_evaluation_summary.json"

with open(summary_path, "w") as f:
    json.dump(evaluation_summary, f, indent=2)

# Verify saved files
assert threshold_path.exists()
assert predictions_path.exists()
assert summary_path.exists()

print("Threshold results:", threshold_path)
print("Validation predictions:", predictions_path)
print("Evaluation summary:", summary_path)

print("\nBaseline evaluation results saved successfully!")

## 10. Candidate Ranking Preparation

The 1D CNN baseline has been trained and evaluated
on the prototype validation dataset.

The next step is to connect fingerprint predictions
to the competition's molecule identification task.

We will:
- Inspect the competition test data
- Check the submission format
- Design a candidate generation strategy
- Rank candidate structures using predicted fingerprints
- Evaluate MRR@25 on held-out molecules

The final ranking must operate at the molecule level,
not only at the individual spectrum level.

In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
from pathlib import Path

import kagglehub
import pandas as pd
import pyarrow.parquet as pq

COMPETITION = "enveda-CASMI26-molecule-id-mass-spectra"

data_dir = Path(
    kagglehub.competition_download(COMPETITION)
)

print("Competition data directory:", data_dir)

print("\nAvailable files:")
for path in sorted(data_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(data_dir))

# Inspect test file
test_files = list(data_dir.rglob("test.parquet"))

if test_files:
    test_pf = pq.ParquetFile(test_files[0])

    print("\nTest rows:", test_pf.metadata.num_rows)
    print("\nTest columns:")
    print(test_pf.schema_arrow.names)

    test_sample = next(
        test_pf.iter_batches(batch_size=3)
    ).to_pandas()

    print("\nTest sample:")
    display(test_sample)

# Inspect submission format
submission_files = list(
    data_dir.rglob("sample_submission.csv")
)

if submission_files:
    sample_submission = pd.read_csv(
        submission_files[0]
    )

    print("\nSubmission shape:", sample_submission.shape)
    print("\nSubmission columns:")
    print(sample_submission.columns.tolist())

    display(sample_submission.head(3))

## 11. Molecule-Level Spectrum Aggregation

The competition test set contains multiple MS/MS spectra
for some molecules.

Our CNN currently predicts one molecular fingerprint
for each individual spectrum.

To obtain molecule-level predictions, we will combine
the fingerprint probabilities from spectra belonging
to the same molecular structure.

For offline validation, we use `inchikey14` as a proxy
for molecule identity because the training data does
not provide `molecule_id`.

For the actual test set, we will group spectra
using the provided `molecule_id`.

Our initial aggregation method is mean pooling.

In [ ]:
# Inspect test molecule grouping
test_info = test_pf.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "precursor_mz"
    ]
).to_pandas()

test_group_sizes = (
    test_info.groupby("molecule_id").size()
)

# Inspect validation structure grouping
val_group_sizes = (
    df_val.groupby("inchikey14").size()
)

print("=== Test Data ===")
print("Test spectra:", len(test_info))
print("Unique molecule IDs:", test_info["molecule_id"].nunique())
print("Spectra per molecule:")
print(test_group_sizes.describe())

print("\n=== Validation Data ===")
print("Validation spectra:", len(df_val))
print("Unique original structure keys:",
      df_val["inchikey14"].nunique())

print("Spectra per structure:")
print(val_group_sizes.describe())

# Check conflicting SMILES representations
smiles_counts = (
    df_val.groupby("inchikey14")["normalized_smiles"]
    .nunique()
)

print(
    "\nValidation structure groups with multiple SMILES:",
    (smiles_counts > 1).sum()
)

In [ ]:
import numpy as np

assert len(val_probs) == len(df_val)

# Convert to NumPy
prob_matrix = val_probs.cpu().numpy()

# Group validation spectra by original structure key
group_indices = (
    df_val.groupby("inchikey14", sort=True).indices
)

molecule_keys = []
molecule_probs = []
spectrum_counts = []

for key, indices in group_indices.items():

    # Probabilities of all spectra in this group
    group_probs = prob_matrix[indices]

    # Mean aggregation
    mean_probs = group_probs.mean(axis=0)

    molecule_keys.append(key)
    molecule_probs.append(mean_probs)
    spectrum_counts.append(len(indices))

molecule_probs = np.stack(
    molecule_probs
).astype(np.float32)

molecule_keys = np.asarray(molecule_keys)
spectrum_counts = np.asarray(spectrum_counts)

print("Molecule-level predictions:", molecule_probs.shape)
print("Unique structure keys:", len(molecule_keys))

print(
    "Structures with multiple spectra:",
    np.sum(spectrum_counts > 1)
)

print(
    "Maximum spectra per structure:",
    spectrum_counts.max()
)

assert molecule_probs.shape == (
    len(molecule_keys), 1024
)

assert np.isfinite(molecule_probs).all()

print("\nMolecule-level aggregation passed!")

## 12. Oracle Candidate Ranking Prototype

We evaluate whether predicted molecular fingerprints
can distinguish candidate molecular structures.

For this initial experiment:

- Candidate structures come from validation molecules.
- Every evaluated query has its correct structure in the candidate library.
- Groups containing multiple SMILES representations are excluded.
- Spectrum-level fingerprint probabilities are aggregated by mean pooling.
- Candidate fingerprints are compared using Soft Tanimoto similarity.

We report MRR@25, Hit@1, Hit@5, and Hit@25.

This is an oracle candidate-ranking benchmark,
not an end-to-end Kaggle competition score.

In [ ]:
# Identify structure groups with a single SMILES representation
smiles_counts = (
    df_val.groupby("inchikey14")["normalized_smiles"]
    .nunique(dropna=False)
)

consistent_keys = set(
    smiles_counts[smiles_counts == 1].index
)

candidate_df = (
    df_val[
        df_val["inchikey14"].isin(consistent_keys)
    ][["inchikey14", "normalized_smiles"]]
    .drop_duplicates(subset=["inchikey14"])
    .sort_values("inchikey14")
    .reset_index(drop=True)
)

# Match each candidate with its aggregated CNN prediction
prob_lookup = dict(zip(molecule_keys, molecule_probs, strict=False))

candidate_keys = []
candidate_fps = []
query_probs = []
invalid_count = 0

for row in candidate_df.itertuples(index=False):
    key = row.inchikey14
    smiles = row.normalized_smiles

    try:
        fp = smiles_to_fingerprint(smiles)
    except (ValueError, TypeError):
        invalid_count += 1
        continue

    candidate_keys.append(key)
    candidate_fps.append(fp)
    query_probs.append(prob_lookup[key])

candidate_keys = np.asarray(candidate_keys)

candidate_fps = np.stack(
    candidate_fps
).astype(np.float32)

query_probs = np.stack(
    query_probs
).astype(np.float32)

print("Original validation groups:", len(molecule_keys))
print("Excluded ambiguous groups:",
      len(molecule_keys) - len(consistent_keys))
print("Invalid candidate fingerprints:", invalid_count)

print("\nCandidate fingerprint matrix:", candidate_fps.shape)
print("Aggregated query predictions:", query_probs.shape)

assert candidate_fps.shape == query_probs.shape
assert candidate_fps.shape[1] == 1024
assert np.isfinite(query_probs).all()

print("\nOracle candidate library ready!")

In [ ]:
# Soft Tanimoto candidate ranking

# Intersection between predicted and candidate fingerprints
intersection = query_probs @ candidate_fps.T

# Sum of fingerprint values
query_sums = query_probs.sum(axis=1, keepdims=True)
candidate_sums = candidate_fps.sum(axis=1)[None, :]

# Soft Tanimoto scores
denominator = query_sums + candidate_sums - intersection

scores = intersection / np.maximum(denominator, 1e-8)

# Rank candidate structures for every query
# Ties are resolved deterministically by candidate order.
ranked_indices = np.argsort(
    -scores,
    axis=1,
    kind="stable"
)

n_queries = len(candidate_keys)

# Find the rank of the correct candidate for each query
inverse_ranks = np.empty_like(ranked_indices)

inverse_ranks[
    np.arange(n_queries)[:, None],
    ranked_indices
] = np.arange(n_queries)[None, :]

# Query i corresponds to candidate i
true_ranks = (
    inverse_ranks[
        np.arange(n_queries),
        np.arange(n_queries)
    ] + 1
)

# Ranking metrics
mrr25 = np.mean(
    np.where(true_ranks <= 25, 1.0 / true_ranks, 0.0)
)

hit1 = np.mean(true_ranks <= 1)
hit5 = np.mean(true_ranks <= 5)
hit25 = np.mean(true_ranks <= 25)

print("=== Oracle Candidate Ranking ===")
print("Queries:", n_queries)
print("Candidates per query:", len(candidate_keys))

print(f"\nMRR@25: {mrr25:.4f}")
print(f"Hit@1:  {hit1:.4f}")
print(f"Hit@5:  {hit5:.4f}")
print(f"Hit@25: {hit25:.4f}")

print("\nMedian correct-candidate rank:",
      np.median(true_ranks))

print("\nRanking evaluation completed!")

## 13. Precursor Mass Consistency Audit

The initial oracle ranking achieved an MRR@25 of 0.0080.

The current ranking uses fingerprint similarity only
and does not consider precursor mass.

We will examine:

- Adduct distributions in training and test spectra
- Theoretical precursor m/z calculated from SMILES
- Mass differences between observed and theoretical m/z
- Whether mass-based candidate filtering is feasible

We first evaluate the common [M+H]+ and [M-H]-
adducts before extending the method to other adducts.

In [ ]:
import numpy as np
import pandas as pd
from rdkit.Chem import Descriptors

# 1. Inspect adduct distribution
print("=== Validation Adduct Distribution ===")
print(df_val["adduct"].value_counts(dropna=False).head(15))

print("\n=== Test Adduct Distribution ===")
print(test_info["adduct"].value_counts(dropna=False).head(15))


# 2. Calculate exact neutral molecular masses
mass_lookup = {}

for row in candidate_df.itertuples(index=False):
    mol = Chem.MolFromSmiles(row.normalized_smiles)

    if mol is not None:
        mass_lookup[row.inchikey14] = Descriptors.ExactMolWt(mol)


# 3. Start with two common adducts
PROTON_MASS = 1.007276466621

ADDUCT_OFFSETS = {
    "[M+H]+": PROTON_MASS,
    "[M-H]-": -PROTON_MASS
}

mass_check = df_val[
    df_val["inchikey14"].isin(mass_lookup)
    & df_val["adduct"].isin(ADDUCT_OFFSETS)
].copy()

mass_check["neutral_mass"] = (
    mass_check["inchikey14"].map(mass_lookup)
)

mass_check["theoretical_mz"] = (
    mass_check["neutral_mass"]
    + mass_check["adduct"].map(ADDUCT_OFFSETS)
)

mass_check["abs_ppm_error"] = (
    (
        mass_check["precursor_mz"]
        - mass_check["theoretical_mz"]
    ).abs()
    / mass_check["theoretical_mz"]
    * 1e6
)

# 4. Summarize consistency
print("\n=== Precursor Mass Consistency ===")

print("Total validation spectra:", len(df_val))
print("Spectra checked:", len(mass_check))

print("\nAbsolute mass error (ppm):")
print(
    mass_check["abs_ppm_error"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

for tolerance in [10, 20, 50, 100]:
    matched = (
        mass_check["abs_ppm_error"] <= tolerance
    ).mean()

    print(
        f"Within {tolerance:3d} ppm: "
        f"{matched:.2%}"
    )

print("\nMass consistency audit completed!")

## 14. Mass-Filtered Oracle Candidate Ranking

The precursor mass consistency audit showed that most
spectra with [M+H]+ or [M-H]- adducts have small mass errors.

We will:

- Inspect spectra with unusually large mass errors
- Estimate neutral mass from precursor m/z and adduct
- Filter candidate molecules by mass tolerance
- Rank the remaining candidates using CNN fingerprint scores
- Compare MRR@25 against the fingerprint-only baseline

We test both 20 ppm and 50 ppm tolerances.

The evaluation retains difficult queries, including
cases where mass filtering removes the correct candidate.

This remains an oracle-library experiment because
the candidate library contains validation structures.

In [ ]:
# Inspect large mass-error records

outliers = mass_check[
    mass_check["abs_ppm_error"] > 20
].copy()

print("=== Mass Error Outliers ===")
print("Spectra checked:", len(mass_check))
print("Spectra above 20 ppm:", len(outliers))
print(
    "Percentage above 20 ppm:",
    f"{len(outliers) / len(mass_check):.2%}"
)

print("\nOutliers by adduct:")
print(outliers["adduct"].value_counts())

print("\nLargest errors:")

display(
    outliers[
        [
            "inchikey14",
            "adduct",
            "precursor_mz",
            "neutral_mass",
            "theoretical_mz",
            "abs_ppm_error"
        ]
    ]
    .sort_values("abs_ppm_error", ascending=False)
    .head(10)
)

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# 1. Calculate neutral masses for oracle candidates
smiles_by_key = (
    candidate_df
    .set_index("inchikey14")["normalized_smiles"]
    .to_dict()
)

candidate_masses = []

for key in candidate_keys:
    mol = Chem.MolFromSmiles(smiles_by_key[key])
    assert mol is not None
    candidate_masses.append(Descriptors.ExactMolWt(mol))

candidate_masses = np.asarray(
    candidate_masses, dtype=np.float64
)

assert len(candidate_masses) == len(candidate_keys)
assert np.all(candidate_masses > 0)


# 2. Estimate neutral mass from observed spectra
# Only use precursor m/z and adduct here, not true SMILES.
observations = df_val[
    ["inchikey14", "precursor_mz", "adduct"]
].copy()

observations["precursor_mz"] = pd.to_numeric(
    observations["precursor_mz"],
    errors="coerce"
)

observations["offset"] = (
    observations["adduct"].map(ADDUCT_OFFSETS)
)

observations["neutral_mass_est"] = (
    observations["precursor_mz"]
    - observations["offset"]
)

valid_observations = observations[
    observations["neutral_mass_est"].notna()
    & np.isfinite(observations["neutral_mass_est"])
    & (observations["neutral_mass_est"] > 0)
]

# Median aggregation across supported spectra
mass_by_key = (
    valid_observations
    .groupby("inchikey14")["neutral_mass_est"]
    .median()
    .to_dict()
)

query_masses = np.asarray(
    [mass_by_key.get(key, np.nan) for key in candidate_keys],
    dtype=np.float64
)

eligible_idx = np.flatnonzero(
    np.isfinite(query_masses)
)

print("Total oracle queries:", len(candidate_keys))
print("Queries with estimated mass:", len(eligible_idx))
print("Queries without estimated mass:",
      len(candidate_keys) - len(eligible_idx))


# 3. Evaluate ranking metrics
def ranking_metrics(ranks):
    ranks = np.asarray(ranks, dtype=np.float64)

    return {
        "MRR@25": np.mean(
            np.where(ranks <= 25, 1.0 / ranks, 0.0)
        ),
        "Hit@1": np.mean(ranks <= 1),
        "Hit@5": np.mean(ranks <= 5),
        "Hit@25": np.mean(ranks <= 25)
    }


# Baseline on exactly the same eligible queries
baseline = ranking_metrics(true_ranks[eligible_idx])

print("\n=== Fingerprint-only Baseline ===")
for name, value in baseline.items():
    print(f"{name}: {value:.4f}")


# 4. Try different mass tolerances
n_candidates = len(candidate_keys)

for tolerance in [20, 50]:

    # Query mass vs candidate neutral mass, in ppm
    ppm_errors = (
        np.abs(
            query_masses[eligible_idx, None]
            - candidate_masses[None, :]
        )
        / candidate_masses[None, :]
        * 1e6
    )

    accepted = ppm_errors <= tolerance

    candidate_counts = accepted.sum(axis=1)

    # Check whether the correct candidate survives
    true_retained = accepted[
        np.arange(len(eligible_idx)),
        eligible_idx
    ]

    # Keep CNN scores only for mass-matched candidates
    filtered_scores = np.where(
        accepted,
        scores[eligible_idx],
        -np.inf
    )

    ranked = np.argsort(
        -filtered_scores,
        axis=1,
        kind="stable"
    )

    # Convert rankings to 1-based positions
    positions = np.empty_like(ranked)

    positions[
        np.arange(len(eligible_idx))[:, None],
        ranked
    ] = np.arange(n_candidates)[None, :]

    correct_ranks = (
        positions[
            np.arange(len(eligible_idx)),
            eligible_idx
        ].astype(np.float64) + 1
    )

    # If mass filtering removed the true candidate,
    # it cannot receive a positive ranking score.
    correct_ranks[~true_retained] = np.inf

    results = ranking_metrics(correct_ranks)

    print(f"\n=== Mass Filtering: {tolerance} ppm ===")
    print("Queries:", len(eligible_idx))
    print("Median candidate count:",
          np.median(candidate_counts))
    print("Queries with zero candidates:",
          np.sum(candidate_counts == 0))
    print("True candidate retained:",
          f"{true_retained.mean():.2%}")

    for name, value in results.items():
        print(f"{name}: {value:.4f}")

print("\nMass-filtered ranking evaluation completed!")

## 15. Mass-Only vs. CNN-Assisted Ranking

The 20 ppm mass filter significantly improved the
oracle ranking results.

However, the median number of remaining candidates
was only one, so high MRR@25 may be largely explained
by precursor mass filtering.

We compare:

1. Mass-only ranking
2. Mass-filtered CNN fingerprint ranking
3. Expected random ranking within the mass filter

We also evaluate queries with multiple remaining
candidates to better measure the contribution of
the CNN model.

In [ ]:
# Candidate count distribution after mass filtering

ppm_errors = (
    np.abs(
        query_masses[eligible_idx, None]
        - candidate_masses[None, :]
    )
    / candidate_masses[None, :]
    * 1e6
)

for tolerance in [20, 50]:

    accepted = ppm_errors <= tolerance
    counts = accepted.sum(axis=1)

    print(f"\n=== {tolerance} ppm Candidate Distribution ===")

    print("Total queries:", len(counts))
    print("Zero candidates:", np.sum(counts == 0))
    print("Exactly 1 candidate:", np.sum(counts == 1))
    print("Exactly 2 candidates:", np.sum(counts == 2))
    print("3–5 candidates:", np.sum((counts >= 3) & (counts <= 5)))
    print("More than 5 candidates:", np.sum(counts > 5))

    print("\nCandidate count percentiles:")
    print(np.percentile(counts, [0, 25, 50, 75, 90, 95, 100]))

In [ ]:
def get_correct_ranks(score_matrix, accepted, eligible_idx):
    """
    Rank candidates using supplied scores.
    Higher scores are better.
    """

    filtered_scores = np.where(
        accepted,
        score_matrix,
        -np.inf
    )

    ranked = np.argsort(
        -filtered_scores,
        axis=1,
        kind="stable"
    )

    positions = np.argsort(
        ranked,
        axis=1,
        kind="stable"
    )

    rows = np.arange(len(eligible_idx))

    ranks = positions[rows, eligible_idx].astype(float) + 1

    retained = accepted[rows, eligible_idx]

    ranks[~retained] = np.inf

    return ranks


for tolerance in [20, 50]:

    accepted = ppm_errors <= tolerance
    candidate_counts = accepted.sum(axis=1)

    rows = np.arange(len(eligible_idx))

    retained = accepted[rows, eligible_idx]

    # 1. Mass-only: smaller ppm error ranks higher
    mass_scores = -ppm_errors

    mass_ranks = get_correct_ranks(
        mass_scores,
        accepted,
        eligible_idx
    )

    # 2. CNN-assisted: fingerprint similarity
    cnn_ranks = get_correct_ranks(
        scores[eligible_idx],
        accepted,
        eligible_idx
    )

    # 3. Expected performance of random ranking
    random_mrr_contributions = np.zeros(
        len(eligible_idx),
        dtype=float
    )

    for i, n in enumerate(candidate_counts):
        if retained[i] and n > 0:
            random_mrr_contributions[i] = (
                sum(1.0 / r for r in range(1, min(n, 25) + 1))
                / n
            )

    print(f"\n=== {tolerance} ppm Ranking Comparison ===")

    print("Mass-only MRR@25:",
          f"{ranking_metrics(mass_ranks)['MRR@25']:.4f}")

    print("Mass + CNN MRR@25:",
          f"{ranking_metrics(cnn_ranks)['MRR@25']:.4f}")

    print("Expected random MRR@25:",
          f"{random_mrr_contributions.mean():.4f}")

    # Evaluate harder queries separately
    ambiguous = candidate_counts >= 2

    print("\nQueries with >= 2 candidates:",
          ambiguous.sum())

    if ambiguous.any():

        print(
            "Mass-only MRR@25 (>=2 candidates):",
            f"{ranking_metrics(mass_ranks[ambiguous])['MRR@25']:.4f}"
        )

        print(
            "Mass + CNN MRR@25 (>=2 candidates):",
            f"{ranking_metrics(cnn_ranks[ambiguous])['MRR@25']:.4f}"
        )

        print(
            "Expected random MRR@25 (>=2 candidates):",
            f"{random_mrr_contributions[ambiguous].mean():.4f}"
        )

print("\nRanking baseline comparison completed!")

## 16. Baseline Ranking Results and Discussion

We compared three ranking strategies using the
oracle validation candidate library:

1. Mass-only ranking
2. Mass-filtered CNN fingerprint ranking
3. Expected random ranking within the mass filter

The results showed that mass-only ranking outperformed
the CNN-assisted approach at both 20 ppm and 50 ppm.

The CNN performed slightly better than random ranking,
suggesting that its predicted fingerprints contain some
structural information. However, this information was not
strong enough to outperform precursor mass accuracy.

The high MRR values are strongly influenced by the small
oracle candidate library, where many queries have only
one or two mass-matched candidates.

This experiment does not represent the final Kaggle score.

Future work will evaluate both CNN architectures using
the same ranking pipeline and a more challenging candidate
library containing more mass-matched molecular structures.

In [ ]:
import json

import numpy as np
import pandas as pd

ranking_results = []

for tolerance in [20, 50]:

    accepted = ppm_errors <= tolerance
    candidate_counts = accepted.sum(axis=1)

    rows = np.arange(len(eligible_idx))
    retained = accepted[rows, eligible_idx]

    # Mass-only
    mass_ranks = get_correct_ranks(
        -ppm_errors,
        accepted,
        eligible_idx
    )

    # CNN-assisted
    cnn_ranks = get_correct_ranks(
        scores[eligible_idx],
        accepted,
        eligible_idx
    )

    # Expected random ranking
    random_rr = np.zeros(len(eligible_idx))

    for i, n in enumerate(candidate_counts):
        if retained[i] and n > 0:
            random_rr[i] = (
                sum(1.0 / r for r in range(1, min(n, 25) + 1))
                / n
            )

    # All eligible queries and harder subset
    for subset_name, mask in [
        ("all_eligible", np.ones(len(eligible_idx), dtype=bool)),
        ("multiple_candidates", candidate_counts >= 2)
    ]:

        mass_mrr = ranking_metrics(
            mass_ranks[mask]
        )["MRR@25"]

        cnn_mrr = ranking_metrics(
            cnn_ranks[mask]
        )["MRR@25"]

        ranking_results.append({
            "model": "CNN1D",
            "tolerance_ppm": tolerance,
            "subset": subset_name,
            "n_queries": int(mask.sum()),
            "mass_only_mrr25": mass_mrr,
            "cnn_mrr25": cnn_mrr,
            "random_expected_mrr25": float(
                random_rr[mask].mean()
            ),
            "true_retention_rate": float(
                retained[mask].mean()
            )
        })

ranking_results_df = pd.DataFrame(ranking_results)

output_path = (
    EVAL_DIR / "cnn1d_ranking_comparison.csv"
)

ranking_results_df.to_csv(output_path, index=False)

display(ranking_results_df.round(4))

print("\nSaved:", output_path)
print("Baseline ranking comparison saved successfully!")